<script src="{{site.baseurl}}/assets/js/code-runner-analytics.js"></script>

## Popcorn Hack 1: Read the API, Then Use It

**What I changed:** the jersey range from `1-99` to `1-55`, and the court size from `144` to `225` square feet.

**Prediction:** `random.randint(1, 55)` can return any whole number from 1 through 55 (both included), so the jersey will be somewhere in that range and will change every run. `math.sqrt(225)` is `15.0`, so the court side prints `15.0 feet`. The average of `[24, 18, 31, 27, 22]` is `24.4`, and `math.ceil(24.4)` rounds up to `25`.

In [ ]:
# CODE_RUNNER: Popcorn Hack 1 - Read the API, then use it (jersey 1-55, bigger court)

import math
import random

# Read the API before calling a procedure
print("random.randint:", random.randint.__doc__)
print("math.ceil:", math.ceil.__doc__)
print()

points_per_game = [24, 18, 31, 27, 22]

# random.choice picks one item out of a list
print("Random game to review:", random.choice(points_per_game), "points")

# CHANGED: random.randint(a, b) now returns a whole number from 1 to 55, including both
jersey = random.randint(1, 55)
print("Random jersey number:", jersey)

# CHANGED: a 225 square foot court instead of 144
side = math.sqrt(225)
print("Square court side:", side, "feet")

# math.ceil rounds up to the next whole number
average = sum(points_per_game) / len(points_per_game)
rounded_up = math.ceil(average)
print("Points per game:", average, "-> rounded up:", rounded_up)

**Actual output:** the docstrings for `randint` and `ceil`, a random game from the list, a jersey number between 1 and 55, `Square court side: 15.0 feet`, and `Points per game: 24.4 -> rounded up: 25`.

The random lines change every run, but stay inside the ranges the API promises. That is the point of reading the docs: I did not need to know how `randint` works inside, only what it accepts and returns.

## Popcorn Hack 2: Write a Flask Route

A route is a procedure with a web address on top. I changed the roster and added my own `/coach` route. Routes are defined above `app.test_client()`.

In [ ]:
# CODE_RUNNER: Popcorn Hack 2 - Flask routes with my own roster and a /coach route

from flask import Flask, jsonify

app = Flask(__name__)

# A route is a procedure with a web address attached to it
@app.route("/score")
def score():
    return jsonify({"home": 58, "away": 52, "quarter": 3})

@app.route("/roster")
def roster():
    return jsonify({"players": ["Maya", "Jordan", "Priya", "Diego"]})

# MY EXTRA ROUTE: /coach
@app.route("/coach")
def coach():
    return jsonify({"coach": "Ms. Rivera", "record": "6-2"})

# test_client visits the addresses without needing a browser
client = app.test_client()

print("/score says:", client.get("/score").get_json())
print("/roster says:", client.get("/roster").get_json())
print("/coach says:", client.get("/coach").get_json())

**Actual output:**

```
/score says: {'away': 52, 'home': 58, 'quarter': 3}
/roster says: {'players': ['Maya', 'Jordan', 'Priya', 'Diego']}
/coach says: {'coach': 'Ms. Rivera', 'record': '6-2'}
```

**My extra route:** `/coach`. (Dictionary key order in the printed output may differ, since `jsonify` can sort keys.)

## Homework Hack: Season Stats Report

A volleyball season instead of the basketball sample, with my own matches and my own next-match date. Three standard library modules: `json`, `statistics`, and `datetime`. The `summarize_season` procedure has a docstring that describes its API, so someone can use it without reading the code.

In [ ]:
# CODE_RUNNER: Homework Hack - Season Stats Report (volleyball)

import json
import statistics
from datetime import date

games = [
    {"date": "2026-09-08", "opponent": "Westview", "points": 21, "minutes": 75},
    {"date": "2026-09-11", "opponent": "Mesa Ridge", "points": 25, "minutes": 82},
    {"date": "2026-09-18", "opponent": "Westview", "points": 19, "minutes": 68},
    {"date": "2026-09-22", "opponent": "Canyon Hills", "points": 27, "minutes": 90},
    {"date": "2026-09-25", "opponent": "Mesa Ridge", "points": 23, "minutes": 79},
]

def summarize_season(season):
    """
    Summarize a list of matches.

    Parameters:
        season: list of dictionaries, each with "date", "opponent", "points", and "minutes"
    Returns:
        dictionary with total_points, average_points, best_game,
        and points_by_opponent
    """
    points = [game["points"] for game in season]

    # Add up points scored against each opponent
    points_by_opponent = {}
    for game in season:
        opponent = game["opponent"]
        points_by_opponent[opponent] = points_by_opponent.get(opponent, 0) + game["points"]

    return {
        "total_points": sum(points),
        "average_points": round(statistics.mean(points), 1),
        "best_game": max(points),
        "points_by_opponent": points_by_opponent,
    }

report = summarize_season(games)

# Subtracting two dates returns a timedelta, and .days is the whole number of days
next_game = date(2026, 10, 20)
report["days_until_next_game"] = (next_game - date.today()).days

print(json.dumps(report, indent=2))

# requirements.txt lists only EXTERNAL packages. json, datetime, and statistics
# come with Python, so they are never listed.
requirements_txt = """pandas
Flask
"""
print()
print("requirements.txt:")
print(requirements_txt)

**Actual output** (the `days_until_next_game` value depends on the day you run it; it is 22 on September 28, 2026):

```json
{
  "total_points": 115,
  "average_points": 23,
  "best_game": 27,
  "points_by_opponent": {
    "Westview": 40,
    "Mesa Ridge": 48,
    "Canyon Hills": 27
  },
  "days_until_next_game": 22
}
```

**`requirements.txt` for a real version of this app:**

```
pandas
Flask
```

`pandas` would hold the season table and `Flask` would serve the scoreboard page. `json`, `datetime`, and `statistics` are **not** listed because they are part of Python's standard library, so they are already installed with Python and never need `pip install`.

## Submission Notes

```
Lesson: CSP 3.14 Libraries
Popcorn 1: what I changed = jersey range 1-55 and court size 225 (sqrt = 15.0)
Popcorn 2: my extra route = /coach
Homework: libraries used = json, statistics, datetime
Homework: days until my next game = 22
```